# Arquitetura MCP: do primeiro servidor ao próximo passo

A ideia é observar uma troca MCP real entre um cliente e um servidor simples. Execute as células em ordem, a partir da raiz deste repositório.

## 1. Quem faz o quê?

- **Host**: aplicação onde a pessoa conversa com um assistente (por exemplo, um editor ou aplicativo de chat).
- **Cliente MCP**: parte do host que se conecta a um servidor, descobre suas capacidades e envia pedidos.
- **Servidor MCP**: programa que anuncia e executa capacidades. Neste exercício é o `main.py`.
- **Modelo**: pode escolher usar uma capacidade oferecida pelo servidor, mas não roda dentro deste servidor.

Fluxo simplificado: `pessoa → host/modelo → cliente MCP ⇄ servidor MCP → tarefas.json`. O protocolo define as mensagens entre cliente e servidor; FastMCP cuida dessa parte para nossas funções Python.

## 2. Transporte e ciclo da conexão

No uso normal, `mcp.run()` inicia o servidor por **stdio**: o host lança o processo e as mensagens MCP passam por entrada e saída padrão. Evite `print()` no servidor nesse modo, porque texto solto em stdout interfere na comunicação. Aqui usamos um cliente **em memória** para experimentar sem configurar um host. A inicialização, a descoberta e as chamadas ainda passam pela interface MCP do FastMCP.

In [ ]:
from pathlib import Path
import sys
import tempfile

from fastmcp import Client

# O notebook fica em python_notebook/ e o servidor na pasta vizinha.
raiz = Path.cwd()
if not (raiz / 'guia_estudos_mcp' / 'main.py').exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz / 'guia_estudos_mcp'))
import main as servidor

# Isola os exercícios: nenhuma tarefa real será alterada.
temporario = tempfile.TemporaryDirectory()
servidor.ARQUIVO = Path(temporario.name) / 'tarefas.json'
cliente = Client(servidor.mcp)


## 3. Descoberta: o cliente vê o contrato

O cliente pode listar **tools**, **resources** e **prompts**. Nomes, descrições e tipos dos argumentos formam o contrato que ajuda um host a apresentar as capacidades e a fazer chamadas válidas. As docstrings das funções aparecem como descrições.

In [ ]:
async with cliente:
    ferramentas = await cliente.list_tools()
    recursos = await cliente.list_resources()
    prompts = await cliente.list_prompts()
    print('Tools:', [item.name for item in ferramentas])
    print('Resources:', [str(item.uri) for item in recursos])
    print('Prompts:', [item.name for item in prompts])


## 4. Tool, resource e prompt na prática

Uma **tool** executa uma ação e pode mudar o estado. Um **resource** entrega conteúdo para leitura; neste exemplo reflete o arquivo JSON. Um **prompt** entrega instruções reutilizáveis ao cliente; não chama um modelo por conta própria. Essas distinções são de intenção e contrato, não uma barreira automática contra efeitos colaterais: o código do servidor ainda é responsável por cumprir o que anuncia.

In [ ]:
async with cliente:
    antes = await cliente.read_resource('estudos://tarefas')
    print('Antes:', antes[0].text)
    resultado = await cliente.call_tool('adicionar_tarefa', {'assunto': 'Ler sobre tools'})
    print('Tool:', resultado.content[0].text)
    depois = await cliente.read_resource('estudos://tarefas')
    print('Depois:', depois[0].text)
    roteiro = await cliente.get_prompt('planejar_estudo', {'tema': 'MCP'})
    print('Prompt:', roteiro.messages[0].content.text)


## 5. Estado, limites e próximo exercício

O JSON demonstra persistência: fechar e abrir o cliente não apaga as tarefas. O arquivo fica no computador que executa o servidor, não no modelo. Neste exemplo de estudo há um único processo e não há autenticação nem controle de acesso. Para atender vários usuários ou gravações simultâneas, seria preciso pensar em armazenamento e permissões adequados.

**Experimente:** chame `concluir_tarefa` com ID 1, releia `estudos://tarefas` e observe o `[x]`. Depois adicione uma tarefa com assunto vazio e veja a validação. Compare as três funções decoradas no `main.py` com o que o cliente descobriu acima.

In [ ]:
async with cliente:
    print((await cliente.call_tool('concluir_tarefa', {'id_tarefa': 1})).content[0].text)
    print((await cliente.read_resource('estudos://tarefas'))[0].text)
    print((await cliente.call_tool('adicionar_tarefa', {'assunto': '  '})).content[0].text)
